# PCA

Principal Component Analysis on California housing features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

data_df = pd.read_csv("housing.csv")
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())
data_df = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

target = data_df['median_house_value'].to_numpy()
X = data_df.drop(columns=['median_house_value']).to_numpy()
feature_names = list(data_df.drop(columns=['median_house_value']).columns)

scaler = StandardScaler()
X = scaler.fit_transform(X)

print(X.shape, len(feature_names))

## Full decomposition

In [ ]:
pca = PCA()
pca.fit(X)

var = pca.explained_variance_ratio_
cum = np.cumsum(var)

plt.figure(figsize=(10, 5))
plt.bar(range(1, len(var) + 1), var, alpha=0.6, label='per-component')
plt.plot(range(1, len(var) + 1), cum, 'o-', color='red', label='cumulative')
plt.axhline(0.9, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('component')
plt.ylabel('variance ratio')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

for i, (v, c) in enumerate(zip(var, cum), 1):
    print(f"PC{i:<2} var={v:.4f} cum={c:.4f}")

## 2D projection

In [ ]:
pca2 = PCA(n_components=2)
Z2 = pca2.fit_transform(X)

plt.figure(figsize=(10, 7))
sc = plt.scatter(Z2[:, 0], Z2[:, 1], c=target, cmap='viridis', s=6, alpha=0.5)
plt.colorbar(sc, label='median_house_value')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(alpha=0.3)
plt.show()

## 3D projection

In [ ]:
pca3 = PCA(n_components=3)
Z3 = pca3.fit_transform(X)

fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(projection='3d')
sc = ax.scatter(Z3[:, 0], Z3[:, 1], Z3[:, 2], c=target, cmap='viridis', s=5, alpha=0.5)
fig.colorbar(sc, label='median_house_value')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
plt.show()

## Component loadings

In [ ]:
loadings = pd.DataFrame(pca.components_[:4].T, index=feature_names,
                        columns=[f'PC{i+1}' for i in range(4)])

fig, axs = plt.subplots(1, 4, figsize=(20, 6), sharey=True)
for i, ax in enumerate(axs):
    loadings.iloc[:, i].sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'PC{i+1}')
    ax.axvline(0, color='k', lw=0.6)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(loadings.round(3))

## Reconstruction error vs k

In [ ]:
ks = list(range(1, X.shape[1] + 1))
errors = []

for k in ks:
    p = PCA(n_components=k)
    Z = p.fit_transform(X)
    X_rec = p.inverse_transform(Z)
    err = np.mean((X - X_rec) ** 2)
    errors.append(err)

plt.figure(figsize=(9, 5))
plt.plot(ks, errors, 'o-')
plt.xlabel('# components')
plt.ylabel('reconstruction MSE')
plt.grid(alpha=0.3)
plt.show()

for k, e in zip(ks, errors):
    print(f"k={k:<2} mse={e:.6f}")

## From-scratch PCA

In [ ]:
class PCAScratch:
    def __init__(self, k):
        self.k = k

    def fit(self, X):
        self.mean = X.mean(axis=0)
        Xc = X - self.mean
        cov = (Xc.T @ Xc) / (Xc.shape[0] - 1)
        vals, vecs = np.linalg.eigh(cov)
        order = np.argsort(vals)[::-1]
        self.components = vecs[:, order[:self.k]].T
        self.explained = vals[order] / vals.sum()
        return self

    def transform(self, X):
        return (X - self.mean) @ self.components.T

scratch = PCAScratch(k=2).fit(X)
Z_scratch = scratch.transform(X)

fig, axs = plt.subplots(1, 2, figsize=(14, 6))
axs[0].scatter(Z2[:, 0], Z2[:, 1], c=target, cmap='viridis', s=5, alpha=0.5)
axs[0].set_title('sklearn')
axs[1].scatter(Z_scratch[:, 0], Z_scratch[:, 1], c=target, cmap='viridis', s=5, alpha=0.5)
axs[1].set_title('from scratch')
for ax in axs:
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

print("Explained variance (scratch top 5):", scratch.explained[:5].round(4))